In [28]:
import h5py

from clearit.config import EMBEDDINGS_DIR, OUTPUTS_DIR, REPO_ROOT, DATASETS_DIR
from clearit.leiden.io import list_patients, load_hdf5_split
from clearit.leiden.preprocess import standardize_and_pca
from clearit.leiden.graph import build_knn_graph
from clearit.leiden.cluster import leiden
from clearit.leiden.exemplars import select_exemplars
from typing import Iterable, List, Optional, Tuple, Dict
import pandas as pd


# Reproducibility
SEED = 42

# Input config with frozen parameters
CONFIG_YAML = REPO_ROOT / "clearit" / "configs" / "leiden_clustering.yaml"

# HDF5 paths
H5_TNBC1 = EMBEDDINGS_DIR / "TNBC1-MxIF8"  / "inForm_MC7"    / "01_features-expressions" / "tnbc1-mxif8.hdf5"
H5_TNBC2 = EMBEDDINGS_DIR / "TNBC2-MIBI44" / "DeepCell_MC17" / "01_features-expressions" / "tnbc2-mibi8.hdf5"

In [7]:
out: Dict[str, List[str]] = {"groups": [], "datasets": []}
with h5py.File(H5_TNBC1, "r") as f:
    def _visit(name, obj):
        if isinstance(obj, h5py.Group):
            out["groups"].append("/" + name if name else "/")
        elif isinstance(obj, h5py.Dataset):
            out["datasets"].append("/" + name)
    f.visititems(_visit)

In [50]:
with h5py.File(H5_TNBC1, "r") as f:
    # List all top-level groups (P01, P02, ...)
    groups = sorted([k for k in f.keys() if k.startswith("P")])
    print(f"Found {len(groups)} groups:\n{groups}\n")

    # Loop through and show info about each "features" dataset
    for g in groups:
        if "features" in f[g]:
            ds = f[g]["features"]
            print(f"{g}/features → shape={ds.shape}, dtype={ds.dtype}")
        else:
            print(f"{g} has no 'features' entry!")
            

Found 62 groups:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26', 'P27', 'P28', 'P29', 'P30', 'P31', 'P32', 'P33', 'P34', 'P35', 'P36', 'P37', 'P38', 'P39', 'P40', 'P41', 'P42', 'P43', 'P44', 'P45', 'P46', 'P47', 'P48', 'P49', 'P50', 'P51', 'P52', 'P53', 'P54', 'P55', 'P56', 'P57', 'P58', 'P59', 'P60', 'P61', 'P62']

P01/features → shape=(27193, 256), dtype=float32
P02/features → shape=(41954, 256), dtype=float32
P03/features → shape=(51054, 256), dtype=float32
P04/features → shape=(31032, 256), dtype=float32
P05/features → shape=(34199, 256), dtype=float32
P06/features → shape=(26532, 256), dtype=float32
P07/features → shape=(65846, 256), dtype=float32
P08/features → shape=(11330, 256), dtype=float32
P09/features → shape=(34268, 256), dtype=float32
P10/features → shape=(61590, 256), dtype=float32
P11/features → shape=(6371, 256), dtype=float32
P12/featu

In [46]:
a

['P01',
 'P02',
 'P03',
 'P04',
 'P05',
 'P06',
 'P07',
 'P08',
 'P09',
 'P10',
 'P11',
 'P12',
 'P13',
 'P14',
 'P15',
 'P16',
 'P17',
 'P18',
 'P19',
 'P20',
 'P21',
 'P22',
 'P23',
 'P24',
 'P25',
 'P26',
 'P27',
 'P28',
 'P29',
 'P30',
 'P31',
 'P32',
 'P33',
 'P34',
 'P35',
 'P36',
 'P37',
 'P38',
 'P39',
 'P40',
 'P41',
 'P42',
 'P43',
 'P44',
 'P45',
 'P46',
 'P47',
 'P48',
 'P49',
 'P50',
 'P51',
 'P52',
 'P53',
 'P54',
 'P55',
 'P56',
 'P57',
 'P58',
 'P59',
 'P60',
 'P61',
 'P62']

In [43]:
df = pd.read_csv(DATASETS_DIR / "TNBC1-MxIF8" / "inForm_MC7" / "labels.csv")
df = df[df["fname"].str.startswith("P02")]

In [47]:
b

<Closed HDF5 dataset>

In [33]:
df

0           True
1           True
2           True
3           True
4           True
           ...  
2270464    False
2270465    False
2270466    False
2270467    False
2270468    False
Name: fname, Length: 2270469, dtype: bool